
# Multi-episode Runner (Notebook)
Run the staged training loop from `run_multi_episode.py` with outputs written to a timestamped `../cachedir/YYYYMMDD_HHMM` outside the code tree.


In [1]:
import sys
from pathlib import Path
from datetime import datetime
import torch

import pandas as pd

# Locate project root even if notebook is launched from elsewhere
ROOT = Path("/home/fit/zhuyingz/WORK/LiuHao/DL-AP")
sys.path.insert(0, str(ROOT))

from config import Config
from training.episode import Episode
from data.simulate_ts import SimulateTS
from experiments import run_utils as utils


In [2]:
print(utils.__file__)

/home/fit/zhuyingz/WORK/LiuHao/DL-AP/experiments/run_utils.py


In [3]:

# Configure run directory and hyperparameters
run_root = ROOT.parent / "cachedir" / datetime.now().strftime("%Y%m%d_%H%M")
base_dir = utils.resolve_base_dir(run_root, ROOT)
utils.ensure_dirs(base_dir)

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
Config.DEVICE = device

# Build components
models = utils.build_models(device)
optimizers = utils.build_optimizers(models)
hyperparams = utils.build_hyperparams()





In [ ]:
# Optionally shrink workload for quick tests
hyperparams.n_paths = 10
hyperparams.epochs = 20
n_episodes = 10
# Run multi-episode loop
stage_order = [
    ("sdf1", ["sdf_fc1"]),
    ("pv", ["policy_value"]),
    ("sdf2", ["sdf_fc1"]),
    ("fc2", ["fc2"]),
]
hyperparams.simulate_horizon = 20
summaries = []
for ep in range(n_episodes):
    episode = Episode(
        models=models,
        optimizers=optimizers,
        config=Config,
        hyperparams=hyperparams,
        device=device,
        episode_id=ep,
    )

    data_kwargs = {
        "n_samples": hyperparams.n_samples,
        "n_paths": hyperparams.n_paths if ep == 0 else min(100, hyperparams.n_paths),
        "group_size": 2 if ep == 0 else Config.SIMULATE_GROUP_SIZE,
        "n_branches": Config.BRANCH_NUM,
    }
    simulate_kwargs = {"horizon": hyperparams.simulate_horizon} if ep > 0 else {}

    ep_summary = {}
    for stage_name, stage_modules in stage_order:
        summary = episode.run_episode(
            n_epochs=hyperparams.epochs,
            batch_size=hyperparams.batch_size,
            log_interval=50,
            train_modules=stage_modules,
            simulate_kwargs=simulate_kwargs,
            **data_kwargs,
        )
        ep_summary[stage_name] = summary.get("module_summaries", summary)
        utils.save_stage_df(ep, stage_name, base_dir, episode.df, episode.df_macro, episode.df_sdf)

    utils.save_models(models, ep, base_dir)

    parent_df = episode.df[episode.df["branch"] < 0] if "branch" in episode.df.columns else episode.df
    ref_state = {
        "eta": 1.0,
        "i": parent_df["i"].median(),
        "x": parent_df["x"].median(),
        "hatcf": parent_df["Hatcf"].median(),
        "lnkf": parent_df["LnKF"].median(),
    }
    utils.plot_surfaces(ep, models["policy_value"], ref_state, device, base_dir)
    utils.plot_distributions(ep, episode.df, models["policy_value"], device, base_dir)
    utils.plot_macro_series(ep, episode.df_macro, base_dir)

    summaries.append(ep_summary)
    print(f"Episode {ep} done")
    print(ep_summary)

print("All episodes done.")
print(summaries)

Simulating paths: 100%|██████████| 50/50 [00:00<00:00, 51.75it/s]


Simulation completed: 1300 firm records, 150 macro records


SDF/FC1 Epoch 50/50: 100%|██████████| 1/1 [00:00<00:00, 125.18it/s]


Episode 0 done
{'sdf1': {'sdf_fc1': {'final_losses': {'sdf': inf, 'total': inf, 'sdf_fc1_grad_norm': 0.0}}}, 'pv': {'policy_value': {'final_losses': {'p0': 0.2277907207608223, 'pi': 0.46812212467193604, 'q': 0.036206260323524475, 'total': 0.7321191132068634, 'policy_value_grad_norm': 0.04048072983106354}}}, 'sdf2': {'sdf_fc1': {'final_losses': {'sdf': inf, 'total': inf, 'sdf_fc1_grad_norm': 0.0}}}, 'fc2': {'fc2': {'final_losses': {'fc2': 9.06264552116394, 'total': 9.06264552116394, 'fc2_grad_norm': 16.333275428517705}}, 'sdf_fc1': {'final_losses': {'sdf': nan, 'total': nan, 'sdf_fc1_grad_norm': 0.0}}}}


Simulating paths: 100%|██████████| 50/50 [19:28<00:00, 23.37s/it]


Simulation completed: 5275000 firm records, 7500 macro records


SDF/FC1 Epoch 1/50:   0%|          | 0/3370 [00:00<?, ?it/s]NaN gradient detected in sdf_fc1
NaN gradient detected in sdf_fc1
NaN gradient detected in sdf_fc1
NaN gradient detected in sdf_fc1
NaN gradient detected in sdf_fc1
NaN gradient detected in sdf_fc1
NaN gradient detected in sdf_fc1
NaN gradient detected in sdf_fc1
NaN gradient detected in sdf_fc1
NaN gradient detected in sdf_fc1
NaN gradient detected in sdf_fc1
NaN gradient detected in sdf_fc1
SDF/FC1 Epoch 1/50:   0%|          | 12/3370 [00:00<00:29, 113.48it/s]NaN gradient detected in sdf_fc1
NaN gradient detected in sdf_fc1
NaN gradient detected in sdf_fc1
NaN gradient detected in sdf_fc1
NaN gradient detected in sdf_fc1
NaN gradient detected in sdf_fc1
NaN gradient detected in sdf_fc1
NaN gradient detected in sdf_fc1
NaN gradient detected in sdf_fc1
NaN gradient detected in sdf_fc1
NaN gradient detected in sdf_fc1
NaN gradient detected in sdf_fc1
NaN gradient detected in sdf_fc1
SDF/FC1 Epoch 1/50:   1%|          | 25/3370 

Simulation completed: 5275000 firm records, 7500 macro records


In [12]:
ep_summary

{'sdf1': {'sdf_fc1': {'final_losses': {'sdf': 3.0581818962527455e-06,
    'total': 3.0581818962527455e-06,
    'sdf_fc1_grad_norm': 0.0038206647230890245}}}}

In [ ]:

# Final simulation with trained models and macro plots
final_sim = SimulateTS(
    models=models,
    config=Config,
    n_paths=hyperparams.n_paths,
    group_size=Config.SIMULATE_GROUP_SIZE,
    branch_num=Config.BRANCH_NUM,
    horizon=hyperparams.simulate_horizon,
    device=device,
)
df_firm_sim, df_macro_sim = final_sim.simulate()
out_dir = base_dir / "data" / "outputs"
out_dir.mkdir(parents=True, exist_ok=True)
df_firm_sim.to_pickle(out_dir / "final_simulate_firm.pkl")
df_macro_sim.to_pickle(out_dir / "final_simulate_macro.pkl")
utils.plot_macro_series(-1, df_macro_sim, base_dir)
print("Final simulation done. Outputs saved to", out_dir)
